# Vector DB Distance Metrics

The 3 operators you configure when creating a vector database index:

| Metric | pgvector | Pinecone | Measures |
|--------|----------|----------|----------|
| Cosine | `<=>` | `metric="cosine"` | Angle between vectors (ignores magnitude) |
| Inner Product | `<#>` | `metric="dotproduct"` | Similarity including magnitude |
| Euclidean (L2) | `<->` | `metric="euclidean"` | Straight-line distance |

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def inner_product(a, b):
    return np.dot(a, b)

def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

## Scenario 1: Cosine — "Same topic, different length"

Two product reviews about battery life:
- Review A: short (3 words)
- Review B: long (same meaning, 10x more words)

In embedding space, longer documents produce vectors with **larger magnitude**
but pointing in the **same direction**.

In [ ]:
# Short review: "great battery life"
review_short = np.array([0.8, 0.6])

# Long review: same topic, 5x more words -> 5x magnitude
review_long = np.array([4.0, 3.0])

# Unrelated review: "terrible screen quality"
review_other = np.array([-0.3, 0.9])

print("=== COSINE (angle only, ignores length) ===")
print(f"  short vs long (same topic):  {cosine_similarity(review_short, review_long):.4f}")
print(f"  short vs other (diff topic): {cosine_similarity(review_short, review_other):.4f}")
print()
print("=== INNER PRODUCT (angle + magnitude) ===")
print(f"  short vs long (same topic):  {inner_product(review_short, review_long):.4f}")
print(f"  short vs other (diff topic): {inner_product(review_short, review_other):.4f}")
print()
print("=== EUCLIDEAN (raw distance) ===")
print(f"  short vs long (same topic):  {euclidean_distance(review_short, review_long):.4f}")
print(f"  short vs other (diff topic): {euclidean_distance(review_short, review_other):.4f}")
print()
print("--- What this tells us ---")
print("Cosine:    short and long reviews are IDENTICAL (1.0) — same meaning")
print("Inner Prod: long review scores 5x higher — magnitude inflates the score")
print("Euclidean: long review is FARTHER away — magnitude pushes them apart")
print()
print(">> For RAG/search: cosine is the right choice here.")
print("   You want 'same meaning' regardless of document length.")

## Scenario 2: Inner Product — "Popularity matters"

Recommendation system where magnitude encodes popularity/confidence:
- Product A: 2 five-star reviews (small magnitude, positive direction)
- Product B: 1000 five-star reviews (large magnitude, same direction)
- Product C: 1000 one-star reviews (large magnitude, opposite direction)

In [ ]:
# User's preference vector (likes electronics, dislikes fashion)
user_pref = np.array([0.9, -0.4])

# Product A: good electronics product, few reviews -> small vector
product_a = np.array([0.3, -0.1])   # 2 reviews

# Product B: good electronics product, many reviews -> large vector  
product_b = np.array([3.0, -1.0])   # 1000 reviews

# Product C: bad fashion product, many reviews -> large but opposite
product_c = np.array([-2.5, 3.0])   # 1000 negative reviews

print("=== COSINE ===")
print(f"  User vs Product A (few reviews):     {cosine_similarity(user_pref, product_a):.4f}")
print(f"  User vs Product B (many reviews):    {cosine_similarity(user_pref, product_b):.4f}")
print(f"  User vs Product C (bad, many reviews):{cosine_similarity(user_pref, product_c):.4f}")
print()
print("=== INNER PRODUCT ===")
print(f"  User vs Product A (few reviews):     {inner_product(user_pref, product_a):.4f}")
print(f"  User vs Product B (many reviews):    {inner_product(user_pref, product_b):.4f}")
print(f"  User vs Product C (bad, many reviews):{inner_product(user_pref, product_c):.4f}")
print()
print("--- What this tells us ---")
print("Cosine:       A and B score the same — cosine can't see popularity")
print("Inner Product: B scores 10x higher than A — popularity boost!")
print("               C scores negative — wrong direction AND many reviews = strong reject")
print()
print(">> For recommendations: inner product is the right choice.")
print("   You want 'similar AND popular/confident' to rank highest.")

## Scenario 3: Euclidean — "How far apart?"

Finding the nearest coffee shop on a map.
Coordinates represent actual physical locations — angle is meaningless here.

In [ ]:
# My location
me = np.array([3.0, 4.0])

# Coffee shops
shop_a = np.array([3.5, 4.2])   # nearby
shop_b = np.array([30.0, 40.0]) # far away
shop_c = np.array([4.0, 3.0])   # nearby but different direction

print("=== EUCLIDEAN (actual distance) ===")
print(f"  Me -> Shop A (nearby):    {euclidean_distance(me, shop_a):.4f}")
print(f"  Me -> Shop B (far):       {euclidean_distance(me, shop_b):.4f}")
print(f"  Me -> Shop C (nearby):    {euclidean_distance(me, shop_c):.4f}")
print()
print("=== COSINE (angle — wrong metric for this!) ===")
print(f"  Me -> Shop A (nearby):    {cosine_similarity(me, shop_a):.4f}")
print(f"  Me -> Shop B (far):       {cosine_similarity(me, shop_b):.4f}")
print(f"  Me -> Shop C (nearby):    {cosine_similarity(me, shop_c):.4f}")
print()
print("--- What this tells us ---")
print("Euclidean: Shop A is closest (0.54), then C (1.41), then B (46.87)")
print("Cosine:    Shop B scores HIGHEST (1.0) — it's in the same direction!")
print("           But it's 47 blocks away. Cosine gives the wrong answer here.")
print()
print(">> For location/spatial search: euclidean is the right choice.")
print("   'Same direction' means nothing when you need 'closest point'.")

## Visual: Why direction != distance

```
         ^  y
         |
    B    |         . Shop B (30, 40)
  (far!) |        /
         |       /  <-- cosine says "perfect match!"
         |      /       (same angle as me)
         |     /
         | .A .Me   <-- euclidean says "A is closest"
         | (3.5,4) (3,4)
         +---.C------------> x
           (4,3)
```

## Side-by-side: Same data, different answers

This is the key insight — the **same vectors** give **different rankings**
depending on which metric you choose.

In [ ]:
# Three document embeddings
query   = np.array([1.0, 0.5])
doc_a   = np.array([2.0, 1.0])    # same direction, 2x magnitude
doc_b   = np.array([1.1, 0.4])    # slightly different direction, similar magnitude
doc_c   = np.array([10.0, 5.0])   # same direction, 10x magnitude

docs = {"Doc A (2x mag)": doc_a, "Doc B (close)": doc_b, "Doc C (10x mag)": doc_c}

print(f"Query vector: {query}")
print()

# Cosine ranking (higher = more similar)
cos_scores = {name: cosine_similarity(query, v) for name, v in docs.items()}
print("COSINE ranking (higher = better):")
for name, score in sorted(cos_scores.items(), key=lambda x: -x[1]):
    print(f"  {score:.4f}  {name}")

print()

# Inner product ranking (higher = more similar)
ip_scores = {name: inner_product(query, v) for name, v in docs.items()}
print("INNER PRODUCT ranking (higher = better):")
for name, score in sorted(ip_scores.items(), key=lambda x: -x[1]):
    print(f"  {score:.4f}  {name}")

print()

# Euclidean ranking (lower = more similar)
euc_scores = {name: euclidean_distance(query, v) for name, v in docs.items()}
print("EUCLIDEAN ranking (lower = better):")
for name, score in sorted(euc_scores.items(), key=lambda x: x[1]):
    print(f"  {score:.4f}  {name}")

print()
print("--- Notice ---")
print("Cosine:   A and C tie (same direction) — B is slightly worse")
print("Inner:    C wins by a landslide (10x magnitude)")
print("Euclidean: B wins (physically closest point)")
print()
print("Same data, 3 different winners. The metric you choose changes the answer.")

## Summary: When to use which

| Use case | Metric | Why |
|----------|--------|-----|
| RAG / text search | **Cosine** `<=>` | "Same meaning" regardless of document length |
| Recommendations | **Inner Product** `<#>` | "Similar AND popular" — magnitude = confidence |
| Spatial / image search | **Euclidean** `<->` | "Closest point" in feature space |

**Rule of thumb:** If your embeddings are **normalized** (length = 1), all three give the same ranking.
Most text embedding models (OpenAI, Cohere) output normalized vectors,
which is why cosine is the default — it's equivalent to the others when vectors are normalized,
and still correct when they're not.